In [1]:
from langgraph.graph import StateGraph,START, END
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from typing import TypedDict
from dotenv import load_dotenv

In [2]:
# model =ChatOpenAI()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [27]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    feedback:str

In [28]:
def create_outline(state:BlogState) -> BlogState:
    #fetch title 
    title = state['title']

    #call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'  
    outline = model.invoke(prompt).content

    
    #update state
    state['outline'] = outline

    return state

In [29]:
def create_blog(state:BlogState) ->BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Wtite a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content
    return state

In [30]:
def evaluate_blog(state:BlogState) -> BlogState:
    outline = state['outline']
    content = state['content']
    prompt = f'Evaluate the following blog content & outline, provide feedback in integer value out of 10  \n {outline},{content}'

    feedback = model.invoke(prompt).content

    state['feedback'] = feedback
    return state

In [31]:
graph = StateGraph(BlogState)

#nodes
graph.add_node("create_outline",create_outline)
graph.add_node("create_blog",create_blog)
graph.add_node("evaluate_blog",evaluate_blog)
#EDGES
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate_blog')
graph.add_edge('evaluate_blog',END)

workflow=graph.compile()

In [32]:
initial_state = {'title':'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '## Blog Outline: The Rise of AI in India - A Transformative Journey\n\n**I. Catchy Title Options:**\n\n*   AI in India: The Dawn of a New Era\n*   India\'s AI Revolution: Opportunities, Challenges, and the Road Ahead\n*   From Data to Dominance: How India is Embracing Artificial Intelligence\n*   The Indian AI Boom: What You Need to Know\n*   Unlocking India\'s Potential: The Power of Artificial Intelligence\n\n**II. Introduction (Approx. 200-300 words)**\n\n*   **Hook:** Start with a compelling statistic, a futuristic scenario, or a thought-provoking question about AI\'s impact on everyday life.\n    *   *Example:* "Imagine a world where personalized healthcare is a reality for every Indian, where farmers predict crop yields with uncanny accuracy, and where traffic flows seamlessly. This isn\'t science fiction; it\'s the promise of Artificial Intelligence, and India is at the forefront of this transformative journey."\n*   **Define AI (brie

In [33]:
print(final_state['outline'])

## Blog Outline: The Rise of AI in India - A Transformative Journey

**I. Catchy Title Options:**

*   AI in India: The Dawn of a New Era
*   India's AI Revolution: Opportunities, Challenges, and the Road Ahead
*   From Data to Dominance: How India is Embracing Artificial Intelligence
*   The Indian AI Boom: What You Need to Know
*   Unlocking India's Potential: The Power of Artificial Intelligence

**II. Introduction (Approx. 200-300 words)**

*   **Hook:** Start with a compelling statistic, a futuristic scenario, or a thought-provoking question about AI's impact on everyday life.
    *   *Example:* "Imagine a world where personalized healthcare is a reality for every Indian, where farmers predict crop yields with uncanny accuracy, and where traffic flows seamlessly. This isn't science fiction; it's the promise of Artificial Intelligence, and India is at the forefront of this transformative journey."
*   **Define AI (briefly):** A simple, accessible explanation for a general audience.

In [34]:
print(final_state['feedback'])

## Blog Content & Outline Evaluation:

**Overall Score: 9/10**

This blog content and outline are exceptionally well-structured, comprehensive, and engaging. It demonstrates a deep understanding of the topic and presents it in a clear, accessible manner for a broad audience. The inclusion of specific examples, potential visuals, and SEO keywords further strengthens its effectiveness.

Here's a breakdown of the strengths and areas for minor improvement:

### Strengths:

*   **Comprehensive Coverage (10/10):** The outline is incredibly thorough, covering all essential aspects of AI in India, from its foundational drivers to specific applications, challenges, and future outlook. It leaves no stone unturned.
*   **Logical Flow and Structure (10/10):** The progression from introduction to conclusion is seamless and intuitive. Each section builds upon the previous one, creating a coherent narrative. The sub-sections within each major point are well-organized and logical.
*   **Engaging Intro

In [36]:
final_state.keys()

dict_keys(['title', 'outline', 'content', 'feedback'])